# Healthcare Predictive Analytics: Diabetes Outcome Classification
**Project type:** Data analysis + supervised machine learning  
**Dataset:** Synthetic demonstration data generated for this repository  
**Target:** `diabetes_outcome` (1 = simulated positive outcome, 0 = simulated negative outcome)

> **Important:** This is an educational project using synthetic data. It is not a diagnostic device and must not be used for medical decisions.

## 1. Business problem
This project explores whether a set of demographic and health-related measurements can classify a synthetic record into one of two outcome groups. The workflow covers data quality, exploratory data analysis, preprocessing, model comparison, evaluation, feature interpretation, and responsible-use limitations.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
df = pd.read_csv("data/healthcare_diabetes_synthetic.csv")
print("Dataset shape:", df.shape)
display(df.head())

## 2. Data quality checks

In [ ]:
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())
print("Outcome distribution:")
display(df["diabetes_outcome"].value_counts().rename(index={0:"Negative (0)",1:"Positive (1)"}).to_frame("records"))
display(df.describe(include="all").T)

## 3. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="diabetes_outcome", ax=axes[0])
axes[0].set_title("Simulated outcome counts")
axes[0].set_xticks([0,1])\naxes[0].set_xticklabels(["Negative (0)", "Positive (1)"])
sns.histplot(data=df, x="glucose", hue="diabetes_outcome", bins=25, element="step",
             stat="density", common_norm=False, ax=axes[1])
axes[1].set_title("Glucose distribution by outcome")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="diabetes_outcome", y="bmi", ax=axes[0])
axes[0].set_title("BMI by simulated outcome")
sns.boxplot(data=df, x="diabetes_outcome", y="age", ax=axes[1])
axes[1].set_title("Age by simulated outcome")
plt.tight_layout()
plt.show()

numeric = df.select_dtypes(include=np.number)
plt.figure(figsize=(9,6))
sns.heatmap(numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric feature correlation matrix")
plt.tight_layout()
plt.show()

## 4. Prepare features and target
Missing numerical values are imputed using training-set medians; categorical values are imputed with the most frequent training value and one-hot encoded. Numeric features are standardized within the pipeline. The test set is held out from fitting.

In [ ]:
target = "diabetes_outcome"
X = df.drop(columns=[target])
y = df[target]
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Positive outcome rate in test set:", round(y_test.mean(), 3))

## 5. Train and compare models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=250, min_samples_leaf=3,
                                            class_weight="balanced", random_state=RANDOM_STATE,
                                            n_jobs=-1)
}
fitted = {}
rows = []
for name, estimator in models.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    scores = pipe.predict_proba(X_test)[:, 1]
    fitted[name] = pipe
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall_sensitivity": recall_score(y_test, pred, zero_division=0),
        "specificity": recall_score(y_test, pred, pos_label=0, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, scores)
    })
results = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
display(results.round(3))

## 6. Evaluate the selected model

In [ ]:
best_name = results.iloc[0]["model"]
best_model = fitted[best_name]
best_pred = best_model.predict(X_test)
print("Selected by test ROC-AUC for demonstration:", best_name)
print(classification_report(y_test, best_pred, target_names=["Negative", "Positive"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, best_pred, display_labels=["Negative","Positive"], cmap="Blues")
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

## 7. Feature importance

In [ ]:
perm = permutation_importance(best_model, X_test, y_test, n_repeats=5,
                              random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1)
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
display(importance)
sns.barplot(data=importance.head(10), x="importance_mean", y="feature")
plt.title("Top predictive features — permutation importance")
plt.xlabel("Mean decrease in ROC-AUC when shuffled")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 8. Ethics, privacy, and limitations
This dataset is synthetic and contains no real patient records. Even with synthetic data, the example does not establish clinical usefulness. A real healthcare model requires validation on representative external data, assessment of subgroup performance, clinical review, privacy protections, and careful monitoring. False negatives and false positives can both have consequences. Do not use this model to diagnose or treat anyone.

## 9. Conclusion
After running the notebook, summarize which model performed best on the held-out test set, what the confusion matrix reveals, and which variables were most predictive in this synthetic dataset. Clearly label all findings as simulated.